In [18]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Load the dataset
data = pd.read_pickle('../../data/processed/cleaned_articles.pkl')

In [19]:
data.info()
data.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39863 entries, 0 to 39862
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   title              39863 non-null  object
 1   publisher          39863 non-null  object
 2   date               39863 non-null  object
 3   section            39863 non-null  object
 4   body               39863 non-null  object
 5   length             39863 non-null  int64 
 6   letters_flag       39863 non-null  bool  
 7   country_flag       39863 non-null  object
 8   foreign_countries  39863 non-null  object
 9   us_mentions        39863 non-null  object
 10  source_file        39863 non-null  object
 11  publisher_raw      39863 non-null  object
 12  section_clean      39845 non-null  object
 13  section_category   34176 non-null  object
dtypes: bool(1), int64(1), object(12)
memory usage: 4.0+ MB


,title,publisher,date,section,body,length,letters_flag,country_flag,foreign_countries,us_mentions,source_file,publisher_raw,section_clean,section_category
0,A Well-Documented Childhood. A Very Private Life.,New York Times,2024-12-31,Section A; Column 0; National Desk; Pg. 16,Jimmy Carter's daughter had an extraordinary a...,1209,False,BOTH,"[Canada, Egypt, Mexico, Nicaragua]","[Massachusetts, Rhode Island, US, Washington]",NYT/1.DOCX,The New York Times,section a; column 0; national desk; pg. 16,US/National
1,These were the big stories in arts and culture...,Other publisher,2024-12-31,WHAT TO KNOW,The arts in Dayton continued to thrive in 2024...,3345,False,BOTH,"[France, Japan, Paris (Capital), Sierra Leone,...","[America, California, Florida, Georgia, Massac...",Other publishers/Files (500) (1).DOCX,Dayton Daily News (Ohio),what to know,None
2,She Exalted The Beauty Of Dance,New York Times,2024-12-31,Section C; Column 0; The Arts/Cultural Desk; P...,She was The New Yorker's first dance critic. H...,1314,False,BOTH,[Male (Capital)],"[New York, North Carolina, US, United States]",NYT/1.DOCX,The New York Times,section c; column 0; the arts/cultural desk; p...,Arts/Culture
3,"Under a Highway in Rio, a Dance Style Charms a...",New York Times,2024-12-31,WORLD; americas,"Trucks, buses and cars rumbled overhead, drown...",1333,False,BOTH,"[Brazil, Lima (Capital)]","[New York, US, United States]",NYT/1.DOCX,The New York Times,world; americas,World/International
4,"Congressional pay, minimum wage stagnant for y...",Other publisher,2024-12-31,OPINION; Pg. A13,ABSTRACT\nMembers of Congress have not seen a ...,921,False,US_ONLY,[],"[America, Delaware, Maryland, New Jersey, New ...",Other publishers/Files (500) (1).DOCX,The Philadelphia Inquirer,opinion; pg. a13,Opinion/Editorial/Letters


In [20]:
print(data.columns)


Index(['title', 'publisher', 'date', 'section', 'body', 'length',
       'letters_flag', 'country_flag', 'foreign_countries', 'us_mentions',
       'source_file', 'publisher_raw', 'section_clean', 'section_category'],
      dtype='object')


In [21]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()  
    # keep letters, space, -
    text = re.sub(r'[^a-z\-\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    # remove stop words
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return ' '.join(words)

data['clean_text'] = data['body'].astype(str).apply(preprocess)

data[['body', 'clean_text']].head()


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jingguo/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /Users/jingguo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,body,clean_text
0,Jimmy Carter's daughter had an extraordinary a...,jimmy carter daughter extraordinary well-docum...
1,The arts in Dayton continued to thrive in 2024...,art dayton continued thrive exciting debut new...
2,She was The New Yorker's first dance critic. H...,new yorker first dance critic wit could devast...
3,"Trucks, buses and cars rumbled overhead, drown...",truck bus car rumbled overhead drowning marcus...
4,ABSTRACT\nMembers of Congress have not seen a ...,abstract member congress seen pay raise since ...


In [27]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_features=50000,     # Limit vocabulary size for efficiency
    ngram_range=(1, 2)      # include unigrams + bigramslike "small business" and "self employment"

)

# Vectorize the clean_text column
X = vectorizer.fit_transform(data['clean_text'])

# Extract vocabulary
vocab = vectorizer.get_feature_names_out()

print("Vocabulary size:", len(vocab))
vocab[:30]   # Show the first 30 vocabulary items


Vocabulary size: 50000


array(['aa', 'aaa', 'aaron', 'aarp', 'aback', 'abaire', 'abandon',
       'abandoned', 'abandoned building', 'abandoned house', 'abandoning',
       'abandonment', 'abatement', 'abbas', 'abbey', 'abbott',
       'abbreviated', 'abby', 'abc', 'abc news', 'abdel', 'abducted',
       'abduction', 'abdul', 'abdullah', 'abe', 'abel', 'aberration',
       'abetted', 'abhorrent'], dtype=object)

In [32]:
# Step 3: Define anchor words 
anchor_words_dict = {
    'Housing': [
        'housing', 'rent', 'tenant', 'affordable', 'homelessness',
        'landlord', 'eviction', 'apartment', 'mortgage'
    ],
    'Education': [
        'education', 'school', 'college', 'teacher', 'university', 'tuition',
        'student', 'curriculum', 'classroom', 'learning'
    ],
    'Union': [
        'union', 'labor', 'strike', 'worker', 'collective',
        'organizing', 'wage', 'bargaining', 'employment'
    ],
    'Election': [
        'election', 'vote', 'campaign', 'party', 'candidate', 'rally',
        'politics', 'ballot', 'primary', 'poll', 'democrat', 'republican'
    ],
    'Military': [
        'military', 'service', 'veteran', 'army', 'soldier',
        'defense', 'troop', 'combat', 'deployment', 'navy'
    ],
    'Indebtedness': [
        'debt', 'struggle', 'credit', 'bankrupt', 'bankruptcy',
        'loan', 'foreclosure', 'repayment', 'interest', 'borrower', 'default', 'financial'
    ],
    'Family': [
        'family', 'child', 'parent', 'marriage', 'spouse', 'household',
        'domestic', 'raising', 'sibling', 'divorce'
    ],
    'Art': [
        'book', 'review', 'theater', 'play', 'movie',
        'culture', 'novel', 'film', 'gallery', 'music', 'performance', 'exhibition', 'artist'
    ],
    'Health': [
        'overdose', 'patient', 'disease', 'treatment', 'mental', 'therapy',
        'nurse', 'clinic', 'pandemic', 'loneliness', 'medicine', 'doctor'
    ],
    'Business Ownership': [
        'business', 'entrepreneur', 'self employment',   
        'client', 'customer', 'firm', 'store', 'small business'   # bigram
    ],
    'Sports': [
        'soccer', 'hockey', 'athlete', 'league', 'tournament', 'game', 'score'
    ],
    'Religion': [
        'faith', 'minister', 'islamic', 'sermon', 'worship', 'belief'
    ]
}

topic_names = list(anchor_words_dict.keys())

# Map words to vocabulary indices
word_to_idx = {w: i for i, w in enumerate(vocab)}

anchor_indices = []
for topic, words in anchor_words_dict.items():
    idx_list = [word_to_idx[w] for w in words if w in word_to_idx]
    anchor_indices.append(idx_list)
    print(f"{topic}: {len(idx_list)} anchors found out of {len(words)} total")


Housing: 9 anchors found out of 9 total
Education: 10 anchors found out of 10 total
Union: 9 anchors found out of 9 total
Election: 12 anchors found out of 12 total
Military: 10 anchors found out of 10 total
Indebtedness: 12 anchors found out of 12 total
Family: 10 anchors found out of 10 total
Art: 13 anchors found out of 13 total
Health: 12 anchors found out of 12 total
Business Ownership: 7 anchors found out of 8 total
Sports: 7 anchors found out of 7 total
Religion: 6 anchors found out of 6 total


In [33]:
# Inspect which anchor words are missing from the vocabulary
missing_anchors = {}

for topic, words in anchor_words_dict.items():
    missing = [w for w in words if w not in word_to_idx]
    if missing:
        missing_anchors[topic] = missing

missing_anchors


{'Business Ownership': ['self employment']}

In [38]:
from corextopic import corextopic as ct

corex = ct.Corex(
    n_hidden=len(topic_names),
    seed=42,
)

corex.fit(X, words=vocab, anchors=anchor_indices)


print("Total correlation (TC) explained:", corex.tc)


Total correlation (TC) explained: 131.97892246781194


In [39]:
for i, topic in enumerate(corex.get_topics(n_words=15)):
    print(f"\n### Topic {i+1}: {topic_names[i]}")
    for word, score, idx in topic:
        print(f"{word:20} {score:.3f}")



### Topic 1: Housing
city                 0.133
housing              0.119
resident             0.113
neighborhood         0.113
building             0.101
apartment            0.086
developer            0.074
york city            0.073
mayor                0.073
community            0.072
property             0.071
brooklyn             0.068
park                 0.067
rent                 0.067
avenue               0.067

### Topic 2: Education
school               0.131
student              0.119
education            0.092
college              0.086
university           0.084
high school          0.082
black                0.064
racial               0.059
teacher              0.052
graduate             0.052
study                0.048
grade                0.047
high                 0.045
public school        0.044
professor            0.042

### Topic 3: Union
policy               0.158
economic             0.126
economy              0.112
state                0.084
government      

In [43]:
from corextopic import corextopic as ct

# Create a new Corex model (no anchor_strength here)
corex2 = ct.Corex(
    n_hidden=len(topic_names),
    seed=42
)

# Fit with anchors and a stronger anchor_strength
corex2.fit(
    X,
    words=vocab,
    anchors=anchor_indices,
    anchor_strength=3 
)

print("TC:", corex2.tc)


TC: 140.78105863187363


In [44]:
for i, topic in enumerate(corex2.get_topics(n_words=15)):
    print(f"\n### Topic {i+1}: {topic_names[i]}")
    for word, score, idx in topic:
        print(f"{word:20} {score:.3f}")



### Topic 1: Housing
housing              0.425
apartment            0.297
rent                 0.242
tenant               0.186
affordable           0.149
city                 0.126
landlord             0.117
resident             0.111
neighborhood         0.109
building             0.099
developer            0.074
community            0.072
property             0.071
york city            0.069
development          0.068

### Topic 2: Education
school               0.631
student              0.519
college              0.362
education            0.354
university           0.341
teacher              0.236
classroom            0.130
high school          0.114
tuition              0.101
learning             0.086
curriculum           0.075
graduate             0.061
high                 0.057
grade                0.053
public school        0.050

### Topic 3: Union
wage                 0.315
worker               0.254
labor                0.230
union                0.202
policy          